In [1]:
from langgraph.graph import StateGraph,START,END
from langchain_google_genai import ChatGoogleGenerativeAI
from typing import TypedDict
from dotenv import load_dotenv

In [2]:
load_dotenv()

True

In [3]:
model = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0
)

In [4]:
class BlogState(TypedDict):
    topic: str
    outline: str
    content: str

In [9]:
def create_outline(state: BlogState) -> BlogState:
    title=state['title']
    prompt=f"Create a detailed outline for a blog post with the title: {title}"
    outline=model.invoke(prompt).content
    state['outline']=outline
    return state

In [8]:
def create_blog(state: BlogState) -> BlogState:
    title=state['title']
    outline=state['outline']
    prompt=f"Write a blog post with the title: {title} and the following outline: {outline}"
    content=model.invoke(prompt).content
    state['content']=content
    return state


In [10]:
graph=StateGraph(BlogState)

graph.add_node('create_outline',create_outline)
graph.add_node('create_blog',create_blog)
graph.add_edge(START,'create_outline')
graph.add_edge('create_outline','create_blog')
graph.add_edge('create_blog',END)
workflow=graph.compile()

In [11]:
initial_state={'title': "What is the capital of France?"}
final_state=workflow.invoke(initial_state)
print(final_state['content'])

KeyError: 'title'